In [ ]:
import os
BUNDLE = {"evals/arc_harness/__init__.py": "\"\"\"ARC-AGI-2 agent harness: prompts, model access, grid checks (more layers to come; see notes/specs).\"\"\"\n", "evals/arc_harness/agent_qwen.py": "\"\"\"Qwen3.8 tool-use agent for ARC-AGI-2 over a raw text-completion API (E012).\n\nBackend-agnostic: the model is any `LLM` with `async complete(prompt, max_tokens, stop) -> Completion`\n(`OpenAICompletionsLLM` covers SGLang / vLLM `/v1/completions`; an offline vLLM engine can implement the same\nmethod). Code runs in an `executor.py` executor (Docker sandbox here, `LocalExecutor` on Kaggle).\n\nWhy raw completions instead of chat + server-side tool parsing:\n- the prompt only ever grows (every past turn is appended verbatim), so the server's prefix cache hits on every\n  turn; the Qwen template keeps the thinking of all turns after the last real user message anyway, so this is\n  exactly what the chat template would render (checked by run_qwen_tools.py --verify-template);\n- a turn cut by max_tokens, or an EOS in the middle of thinking, is continued in place;\n- tool calls are parsed here (`parse_tool_calls`), so parse failures are visible and get feedback;\n- no dependence on a particular server's tool / reasoning parser.\n\nHarness logic follows E009 sub80s40 (DeepSeek's best): grid mode, `lean2sub` prompt, `run_python` +\n`submit_answer` tools, budget note on every tool output with a soft target and hard cap, and at the hard cap a\nforced `submit_answer` call (the model only writes the expression) with the session `answer` as fallback.\nExactly one scored grid per attempt; `answer_source` is submit | forced | parse | lenient | session | None.\n\"\"\"\nimport asyncio\nimport json\nimport re\nimport time\nfrom dataclasses import asdict, dataclass\n\nfrom .arcprize import build_prompt, parse\nfrom .executor import check_answer_on_train, submit_answer, tool_output\nfrom .grids import valid_grid\nfrom .prompts import PROMPT_SUFFIXES\n\nHARNESS_VERSION = \"qwen-tools-2\"  # 2: forced-submit text names the expression to use\nCONTEXT = 262144\nTHINK_CLOSE = \"</think>\"\nGEN_PROMPT = \"<|im_start|>assistant\\n<think>\\n\"\nTIER_SYSTEM = {\n    \"xhigh\": \"Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, \"\n             \"consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final \"\n             \"answer.\",\n    \"medium\": None,\n    \"low\": \"Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion \"\n           \"without unnecessary elaboration.\",\n}\n# E009 lean2sub: lean2 + the submit_answer hand-off (byte-identical to run_deepseek_tools.py).\n_SUBMIT = (\"\\n\\nWhen your code reproduces every training output and its assumptions hold on test_input, call \"\n           \"submit_answer with an expression such as `solve(test_input)`: the harness evaluates it in your session \"\n           \"and that grid is your answer, so never retype the grid. Keep your current best guess for the test \"\n           \"output in a variable `answer`; if you run out of budget it is submitted automatically.\")\nPROMPTS = {\"lean2sub\": PROMPT_SUFFIXES[\"lean2\"] + _SUBMIT}\nTOOLS_HEADER = (\"# Tools\\n\\nYou have access to the following functions:\\n\\n<tools>\")\nTOOLS_FOOTER = (\n    \"\\n</tools>\\n\\nIf you choose to call a function ONLY reply in the following format with NO suffix:\\n\\n\"\n    \"<tool_call>\\n<function=example_function_name>\\n<parameter=example_parameter_1>\\nvalue_1\\n</parameter>\\n\"\n    \"<parameter=example_parameter_2>\\nThis is the value for the second parameter\\nthat can span\\nmultiple lines\\n\"\n    \"</parameter>\\n</function>\\n</tool_call>\\n\\n<IMPORTANT>\\nReminder:\\n- Function calls MUST follow the specified \"\n    \"format: an inner <function=...></function> block must be nested within <tool_call></tool_call> XML tags\\n\"\n    \"- Required parameters MUST be specified\\n- You may provide optional reasoning for your function call in \"\n    \"natural language BEFORE the function call, but NOT after\\n- If there is no function call available, answer \"\n    \"the question like normal with your current knowledge and do not tell the user about function calls\\n\"\n    \"</IMPORTANT>\")\n# At the hard cap: close the thinking and open a submit_answer call; the model writes only the expression.\nFORCE_THINK = (\"\\n\\nI am out of budget, so I must submit my best answer now as one short expression: `answer` if I \"\n               \"set it, otherwise solve(test_input) or the name of my best function applied to test_input.\\n</think>\\n\\n\")\nFORCE_CALL = \"<tool_call>\\n<function=submit_answer>\\n<parameter=code>\\n\"\nLOW_YIELD = 20\n\n\n@dataclass\nclass AgentConfig:\n    tier: str = \"xhigh\"\n    prompt: str = \"lean2sub\"\n    max_total_output: int = 80000   # hard cap: generated tokens per attempt, all turns\n    budget_soft: int = 40000        # soft target stated in the budget notes (0 = none)\n    budget_notes: float = 0.01      # notes on tool outputs once this fraction of the hard cap is used (0 = off)\n    budget_warn: float = 0.75       # from this fraction on, the note says to submit now\n    soft_wording: str = \"gentle\"    # past the soft target: gentle = submit once a rule fits; strong = submit now (E010)\n    chunk_tokens: int = 16384       # max tokens per completion request (a turn may take several)\n    max_turns: int = 60\n    tool_timeout: float = 30\n    max_tool_output: int = 16000\n    verify_submit: bool = False     # E011: re-run the submitted expression on train inputs; reject once on mismatch\n    force_submit_tokens: int = 256  # tokens for the forced submit_answer expression at the cap (0 = off)\n    temperature: float = 1.0\n    top_p: float = 0.95\n    top_k: int = 20\n    max_eos_continues: int = 4      # EOS inside thinking: continue the turn at most this many times\n\n\n# ---------------------------------------------------------------- rendering (Qwen3.8 chat template, tools on)\n\ndef tool_specs(cfg):\n    descriptions = {\n        \"run_python\": (\n            \"Execute a Python cell in your persistent IPython session. Variables/functions persist. \"\n            \"Preloaded: train (list of {'input': grid, 'output': grid}), test_input (grid), numpy as np. \"\n            f\"scipy is available. No internet or host files. Timeout {cfg.tool_timeout:g}s. \"\n            \"Use print() to inspect results. Check transformations on all training pairs.\"),\n        \"submit_answer\": (\n            \"End this attempt with your test output. `code` is a Python expression evaluated in your session, \"\n            \"e.g. solve(test_input) or answer, giving a rectangular grid of ints 0-9 (list or numpy array); \"\n            \"statements that set a variable `answer` also work. The harness reads the grid directly: do not \"\n            \"retype it. Invalid grids return an error and the attempt continues.\"),\n    }\n    return [{\"type\": \"function\", \"function\": {\n        \"name\": name, \"description\": description,\n        \"parameters\": {\"type\": \"object\", \"properties\": {\"code\": {\"type\": \"string\"}},\n                       \"required\": [\"code\"], \"additionalProperties\": False},\n    }} for name, description in descriptions.items()]\n\n\ndef render_system(tier, tools):\n    s = \"<|im_start|>system\\n\"\n    if TIER_SYSTEM[tier]:\n        s += TIER_SYSTEM[tier] + \"\\n\\n\"\n    s += TOOLS_HEADER + \"\".join(\"\\n\" + json.dumps(t, ensure_ascii=False) for t in tools) + TOOLS_FOOTER\n    return s + \"<|im_end|>\\n\"\n\n\ndef render_user(content):\n    return f\"<|im_start|>user\\n{content.strip()}<|im_end|>\\n\"\n\n\ndef render_tool_responses(outputs):\n    return \"<|im_start|>user\" + \"\".join(f\"\\n<tool_response>\\n{o.strip()}\\n</tool_response>\" for o in outputs) + \\\n        \"<|im_end|>\\n\"\n\n\ndef render_assistant(reasoning, content, calls):\n    \"\"\"A past assistant turn as the chat template renders it (used to check our verbatim appends).\"\"\"\n    s = f\"<|im_start|>assistant\\n<think>\\n{reasoning.strip()}\\n</think>\\n\\n{content.strip()}\"\n    for i, (name, params) in enumerate(calls):\n        s += (\"\\n\\n\" if content.strip() else \"\") if i == 0 else \"\\n\"\n        s += f\"<tool_call>\\n<function={name}>\\n\"\n        s += \"\".join(f\"<parameter={k}>\\n{v}\\n</parameter>\\n\" for k, v in params.items())\n        s += \"</function>\\n</tool_call>\"\n    return s + \"<|im_end|>\\n\"\n\n\ndef user_content(train, test_input, prompt):\n    return build_prompt(train, test_input) + PROMPTS[prompt]\n\n\ndef initial_prompt(train, test_input, cfg):\n    return render_system(cfg.tier, tool_specs(cfg)) + render_user(user_content(train, test_input, cfg.prompt)) \\\n        + GEN_PROMPT\n\n\n# ---------------------------------------------------------------- parsing\n\n_CALL = re.compile(r\"<tool_call>(.*?)</tool_call>\", re.DOTALL)\n_FUNC = re.compile(r\"^\\s*<function=([^>\\n]+)>(.*?)</function>\\s*$\", re.DOTALL)\n_PARAM = re.compile(r\"<parameter=([^>\\n]+)>\\n?(.*?)\\n?</parameter>\", re.DOTALL)\n\n\ndef split_turn(text):\n    \"\"\"(reasoning, content) of one generated assistant turn; content is None while still thinking.\"\"\"\n    i = text.find(THINK_CLOSE)\n    if i < 0:\n        return text, None\n    return text[:i], text[i + len(THINK_CLOSE):]\n\n\ndef parse_tool_calls(content):\n    \"\"\"Tool calls in the answer part of a turn: (calls, errors). calls = [(name, {param: str})].\n\n    errors lists malformed blocks (unclosed <tool_call>, missing <function=...>).\"\"\"\n    calls, errors = [], []\n    for block in _CALL.findall(content):\n        m = _FUNC.match(block)\n        if not m:\n            errors.append(\"malformed <tool_call> block: expected <function=NAME>...</function>\")\n            continue\n        calls.append((m.group(1).strip(), {k.strip(): v for k, v in _PARAM.findall(m.group(2))}))\n    tail = _CALL.sub(\"\", content)\n    if \"<tool_call>\" in tail or \"<function=\" in tail:\n        errors.append(\"unclosed or malformed tool call: use <tool_call><function=NAME><parameter=code>...\"\n                      \"</parameter></function></tool_call>\")\n    return calls, errors\n\n\ndef lenient_parse(text):\n    \"\"\"Last valid JSON grid in the text (trailing commas tolerated), else the last block of one-row-per-line.\"\"\"\n    text = text or \"\"\n    for start in [i for i, ch in enumerate(text) if ch == \"[\"][::-1]:\n        if not re.match(r\"\\[\\s*\\[\", text[start:start + 50]):\n            continue\n        depth = 0\n        for j in range(start, len(text)):\n            depth += (text[j] == \"[\") - (text[j] == \"]\")\n            if depth == 0:\n                block = text[start:j + 1]\n                for candidate in (block, re.sub(r\",\\s*\\]\", \"]\", block)):\n                    try:\n                        grid = json.loads(candidate)\n                    except json.JSONDecodeError:\n                        continue\n                    if valid_grid(grid):\n                        return grid\n                break\n    rows, best = [], None\n    for line in text.splitlines() + [\"\"]:\n        line = line.strip().rstrip(\",\")\n        try:\n            row = json.loads(line) if line.startswith(\"[\") else None\n        except json.JSONDecodeError:\n            row = None\n        if isinstance(row, list) and row and all(type(v) is int for v in row):\n            rows.append(row)\n            continue\n        if rows and valid_grid(rows):\n            best = rows\n        rows = []\n    return best\n\n\ndef budget_note(cfg, used):\n    \"\"\"Status line appended to the last tool output of a turn (E009 wording).\"\"\"\n    if not cfg.budget_notes or used < cfg.budget_notes * cfg.max_total_output:\n        return \"\"\n    hard = cfg.max_total_output\n    if cfg.budget_soft:\n        note = (f\"\\n[budget: {used / 1000:.0f}k output tokens used; aim to finish within {cfg.budget_soft / 1000:.0f}k \"\n                f\"(hard limit {hard / 1000:.0f}k)]\")\n        if cfg.budget_soft <= used < cfg.budget_warn * hard:\n            note += (\" Past the target: submit your best answer now with submit_answer, unless one more experiment \"\n                     \"will very likely make every training pair pass.\" if cfg.soft_wording == \"strong\"\n                     else \" Past the target: submit as soon as a rule fits every training pair.\")\n    else:\n        note = f\"\\n[budget: {used / 1000:.0f}k of {hard / 1000:.0f}k output tokens used]\"\n    if used >= cfg.budget_warn * hard:\n        note += \" Budget nearly spent: submit your best answer now with submit_answer (or at least set `answer`).\"\n    return note\n\n\n# ---------------------------------------------------------------- model backend\n\n@dataclass\nclass Completion:\n    text: str\n    completion_tokens: int\n    prompt_tokens: int\n    finish: str  # \"stop\" (EOS or stop string) or \"length\"\n\n\nclass OpenAICompletionsLLM:\n    \"\"\"Raw /v1/completions on an OpenAI-compatible server (SGLang, vLLM). Retries transient errors only:\n    a completion has no side effects, so a retry is safe.\"\"\"\n\n    def __init__(self, base_url, api_key, model, temperature=1.0, top_p=0.95, top_k=20, timeout=3600,\n                 max_retries=8, extra_body=None):\n        from openai import AsyncOpenAI\n        self.client = AsyncOpenAI(api_key=api_key or \"none\", base_url=base_url, timeout=timeout, max_retries=0)\n        self.model, self.temperature, self.top_p, self.top_k = model, temperature, top_p, top_k\n        self.max_retries, self.extra_body = max_retries, extra_body or {}\n        self.transient_errors = 0\n\n    async def complete(self, prompt, max_tokens, stop=None):\n        import openai\n        errs = 0\n        while True:\n            try:\n                r = await self.client.completions.create(\n                    model=self.model, prompt=prompt, max_tokens=max_tokens, temperature=self.temperature,\n                    top_p=self.top_p, stop=stop, extra_body={\"top_k\": self.top_k, **self.extra_body})\n                c = r.choices[0]\n                return Completion(c.text, r.usage.completion_tokens, r.usage.prompt_tokens, c.finish_reason)\n            except (openai.APIConnectionError, openai.APITimeoutError, openai.APIStatusError) as exc:\n                status = getattr(exc, \"status_code\", None)\n                if status is not None and status < 500 and status not in (408, 429):\n                    raise\n                errs += 1\n                self.transient_errors += 1\n                if errs > self.max_retries:\n                    raise\n                await asyncio.sleep(min(60, 5 * 2 ** min(errs - 1, 4)))\n\n    async def close(self):\n        await self.client.close()\n\n\n# ---------------------------------------------------------------- agent loop\n\nasync def _call(fn, *args):\n    # Blocking executor calls run in the loop's default thread pool. With many concurrent attempts, set a pool with\n    # one thread per attempt: loop.set_default_executor(ThreadPoolExecutor(concurrency + 4)).\n    return await asyncio.to_thread(fn, *args)\n\n\nasync def solve(llm, make_executor, train, test_input, cfg):\n    \"\"\"One attempt. Returns (transcript, result). transcript is the full raw prompt+generation text.\n\n    make_executor(train, test_input) -> executor (context manager with .execute(code)); train holds the\n    training pairs with outputs (used host-side for --verify-submit), test_input the test grid only.\"\"\"\n    prompt = initial_prompt(train, test_input, cfg)\n    used, turns, n_calls, n_tool_err, n_parse_err, n_eos, n_chunks = 0, 0, 0, 0, 0, 0, 0\n    pred = source = stop = error = None\n    max_prompt, content, submit_verified, forced_codes = 0, \"\", None, set()\n    turn_log, t0 = [], time.monotonic()\n    session = None\n    try:\n        session = make_executor(train, test_input)\n        await _call(session.__enter__)\n        while True:\n            if turns >= cfg.max_turns:\n                stop = \"max_turns\"\n                break\n            turns += 1\n            text, low_streak, after_eos, out_of_budget = \"\", 0, False, False\n            turn_tokens = 0\n            while True:  # one assistant turn, possibly over several requests\n                left = cfg.max_total_output - used\n                room = CONTEXT - max_prompt - turn_tokens - 512\n                if left <= 0 or room <= 0:\n                    out_of_budget = True\n                    break\n                c = await llm.complete(prompt + text, min(cfg.chunk_tokens, left, room))\n                n_chunks += 1\n                used += c.completion_tokens\n                turn_tokens += c.completion_tokens\n                max_prompt = max(max_prompt, c.prompt_tokens)\n                text += c.text\n                if after_eos:\n                    low_streak = low_streak + 1 if c.completion_tokens < LOW_YIELD else 0\n                after_eos = False\n                if c.finish == \"length\":\n                    continue\n                if THINK_CLOSE not in text and n_eos < cfg.max_eos_continues and low_streak < 2:\n                    n_eos += 1  # EOS inside thinking: let it carry on\n                    after_eos = True\n                    continue\n                break\n            reasoning, answer_part = split_turn(text)\n            entry = {\"turn\": turns, \"tokens\": turn_tokens, \"closed_think\": answer_part is not None}\n            turn_log.append(entry)\n            if out_of_budget or answer_part is None:\n                # Hard cap (or thinking that never closes): force a submit_answer call, then the session answer.\n                stop = \"output_budget\" if out_of_budget else \"unclosed_think\"\n                if cfg.force_submit_tokens:\n                    base = text + FORCE_THINK if answer_part is None else text[:len(text) - len(answer_part)] + \"\\n\\n\"\n                    forced = base + FORCE_CALL\n                    c = await llm.complete(prompt + forced, cfg.force_submit_tokens, stop=[\"</parameter>\"])\n                    n_chunks += 1\n                    used += c.completion_tokens\n                    expr = c.text.strip()\n                    entry[\"forced_expr\"] = expr[:200]\n                    text = forced + c.text + \"\\n</parameter>\\n</function>\\n</tool_call>\"\n                    if expr:\n                        out, grid = await _call(submit_answer, session, expr)\n                        entry[\"forced_result\"] = out[:200]\n                        if grid is not None:\n                            pred, source = grid, \"forced\"\n                prompt += text\n                break\n            calls, perrs = parse_tool_calls(answer_part)\n            if not calls and not perrs:  # a final text reply\n                content = answer_part\n                candidate = parse(answer_part)\n                if valid_grid(candidate):\n                    pred, source = candidate, \"parse\"\n                else:\n                    candidate = lenient_parse(answer_part)\n                    if candidate is not None:\n                        pred, source = candidate, \"lenient\"\n                prompt += text + \"<|im_end|>\\n\"\n                stop = \"final\"\n                break\n            outputs, accepted = [], None\n            n_parse_err += len(perrs)\n            for name, params in calls:\n                n_calls += 1\n                code = params.get(\"code\")\n                if accepted is not None:\n                    outputs.append(\"[skipped: answer already accepted]\")\n                    continue\n                if not isinstance(code, str) or name not in (\"run_python\", \"submit_answer\"):\n                    n_parse_err += 1\n                    n_tool_err += 1\n                    outputs.append(f\"[tool error: unknown tool {name!r} or missing `code` parameter; tools are \"\n                                   \"run_python(code) and submit_answer(code)]\")\n                    continue\n                if name == \"run_python\":\n                    result = await _call(session.execute, code)\n                    n_tool_err += result[\"process_status\"] != \"completed\"\n                    outputs.append(tool_output(result, cfg.max_tool_output))\n                    continue\n                out, grid = await _call(submit_answer, session, code)\n                if grid is not None and cfg.verify_submit:\n                    check = await _call(check_answer_on_train, session, code)\n                    if check and check[0] and code not in forced_codes:\n                        forced_codes.add(code)\n                        out, grid = (f\"[answer not accepted yet: the same expression with test_input = each \"\n                                     f\"training input fails training pairs {check[0]} ({check[1]}). Fix the rule, \"\n                                     \"or call submit_answer again with exactly the same code to submit anyway.]\"), None\n                    else:\n                        submit_verified = None if check is None else not check[0]\n                n_tool_err += grid is None\n                outputs.append(out)\n                if grid is not None:\n                    accepted = grid\n            for e in perrs:\n                n_tool_err += 1\n                outputs.append(f\"[tool error: {e}]\")\n            entry[\"calls\"] = [name for name, _ in calls] + [\"<malformed>\"] * len(perrs)\n            if accepted is not None:\n                pred, source, stop = accepted, \"submit\", \"submitted\"\n                prompt += text + \"<|im_end|>\\n\"\n                break\n            outputs[-1] += budget_note(cfg, used)\n            prompt += text + \"<|im_end|>\\n\" + render_tool_responses(outputs) + GEN_PROMPT\n        if pred is None and session is not None:\n            out, grid = await _call(submit_answer, session, \"answer\")\n            if grid is not None:\n                pred, source = grid, \"session\"\n    except Exception as exc:\n        error, stop = f\"{type(exc).__name__}: {exc}\"[:1000], \"error\"\n    finally:\n        if session is not None:\n            try:\n                await _call(session.close)\n            except Exception:\n                pass\n    result = {\"pred\": pred, \"answer_source\": source if pred is not None else None, \"content\": content[-2000:],\n              \"stop_reason\": stop, \"turns\": turns, \"tool_calls\": n_calls, \"tool_errors\": n_tool_err,\n              \"parse_failures\": n_parse_err, \"eos_continues\": n_eos, \"chunks\": n_chunks,\n              \"usage\": {\"completion\": used, \"max_prompt\": max_prompt}, \"turn_log\": turn_log,\n              \"seconds\": round(time.monotonic() - t0, 1)}\n    if cfg.verify_submit:\n        result[\"submit_verified\"] = submit_verified if source == \"submit\" else None\n        result[\"submit_rejections\"] = len(forced_codes)\n    if error:\n        result[\"error\"] = error\n    return prompt, result\n\n\ndef config_dict(cfg, model):\n    d = {**asdict(cfg), \"model\": model, \"harness_version\": HARNESS_VERSION}\n    if d[\"soft_wording\"] == \"gentle\":  # keep configs of runs made before the option existed unchanged\n        del d[\"soft_wording\"]\n    return d\n", "evals/arc_harness/executor.py": "\"\"\"Code-execution backends for the tool harness, behind one small interface.\n\nAn executor is a persistent Python session for one attempt:\n\n    with make_executor(train, test_input) as ex:     # train: [{'input', 'output'}], test_input: grid\n        result = ex.execute(code)                    # {'process_status': completed|error|timeout,\n                                                     #  'stdout': str, 'stderr': str, ['state_reset': True]}\n\nIt preloads `train`, `test_input`, `np` and `json`, keeps variables between cells, and shows the value of a\ncell's last expression (IPython-style). Attributes used by the answer helpers: `train`, `test_input`, `max_output`.\n\nBackends:\n- `SandboxSession` (evals/sandbox_client.py): the Docker NeMo-Skills sandbox (no network, no host mounts).\n- `LocalExecutor` (here): a child Python process per attempt, for Kaggle where Docker is unavailable. It is not a\n  security boundary (same user, same filesystem); it only adds a memory limit, a timeout and output truncation.\n\n`submit_answer` / `check_answer_on_train` / `tool_output` are kept identical to evals/run_deepseek_tools.py and\nevals/sandbox_client.py (copied, so this package runs without those scripts).\n\"\"\"\nimport json\nimport os\nimport select\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom uuid import uuid4\n\nfrom .grids import valid_grid\n\n# ---------------------------------------------------------------- local executor\n\n_CHILD = r'''\nimport ast, contextlib, io, json, os, sys, traceback\nresp = os.fdopen(int(sys.argv[1]), \"w\")\nlimit = int(sys.argv[2])\nns = {\"__name__\": \"__main__\"}\n\ndef run(code):\n    tree = ast.parse(code, \"<cell>\", \"exec\")\n    last = None\n    if tree.body and isinstance(tree.body[-1], ast.Expr):\n        last = ast.Expression(tree.body.pop().value)\n    exec(compile(tree, \"<cell>\", \"exec\"), ns)\n    if last is not None:\n        value = eval(compile(last, \"<cell>\", \"eval\"), ns)\n        if value is not None:\n            print(repr(value))\n\ndef plain_traceback(exc):\n    frames = [f for f in traceback.extract_tb(exc.__traceback__) if f.filename != \"<string>\"]\n    text = \"\".join(traceback.format_list(frames)) if frames else \"\"\n    return \"Traceback (most recent call last):\\n\" + text + \"\".join(traceback.format_exception_only(type(exc), exc))\n\nfor line in sys.stdin:\n    req = json.loads(line)\n    out, err = io.StringIO(), io.StringIO()\n    status = \"completed\"\n    try:\n        with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):\n            run(req[\"code\"])\n    except KeyboardInterrupt:\n        status = \"timeout\"\n        err.write(\"KeyboardInterrupt: cell timed out (state kept)\\n\")\n    except BaseException as exc:\n        status = \"error\"\n        try:\n            err.write(plain_traceback(exc))\n        except BaseException:\n            err.write(f\"{type(exc).__name__}: {exc}\\n\")\n    o, e = out.getvalue(), err.getvalue()\n    if len(o) > limit:\n        o = o[:limit] + \"\\n[output truncated]\"\n    if len(e) > limit:\n        e = e[:limit] + \"\\n[output truncated]\"\n    resp.write(json.dumps({\"process_status\": status, \"stdout\": o, \"stderr\": e}) + \"\\n\")\n    resp.flush()\n'''\n\n\ndef _limit_memory(nbytes):\n    def apply():\n        import resource\n        os.setsid()  # own process group: a timeout kill also takes children the cell spawned\n        if nbytes:\n            resource.setrlimit(resource.RLIMIT_AS, (nbytes, nbytes))\n    return apply\n\n\nclass LocalExecutor:\n    \"\"\"Persistent Python in a child process. On timeout the cell gets SIGINT (state kept, like the patched\n    sandbox); if it does not stop within `grace` seconds the process is killed and restarted with only the\n    task data (`state_reset`).\"\"\"\n\n    def __init__(self, train, test_input, timeout=30, max_output=16000, memory_mb=4096, grace=5,\n                 python=sys.executable):\n        self.train, self.test_input = train, test_input\n        self.timeout, self.max_output, self.memory_mb, self.grace = timeout, max_output, memory_mb, grace\n        self.python = python\n        self.proc = self.resp = None\n        self.closed = False\n        data = json.dumps({\"train\": train, \"test_input\": test_input})\n        self.prelude = (\n            \"import json, numpy as np\\n\"\n            f\"_arc_data = json.loads({data!r})\\n\"\n            \"train, test_input = _arc_data['train'], _arc_data['test_input']\\n\"\n            \"del _arc_data\\n\"\n        )\n\n    def _spawn(self):\n        r, w = os.pipe()\n        self.proc = subprocess.Popen(\n            [self.python, \"-I\", \"-u\", \"-c\", _CHILD, str(w), str(self.max_output)],\n            stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, pass_fds=(w,),\n            preexec_fn=_limit_memory(self.memory_mb * 2 ** 20), text=True, cwd=\"/tmp\",\n            env={\"PATH\": os.environ.get(\"PATH\", \"\"), \"OMP_NUM_THREADS\": \"1\", \"OPENBLAS_NUM_THREADS\": \"1\",\n                 \"MKL_NUM_THREADS\": \"1\"})\n        os.close(w)\n        self.resp, self.buf = r, b\"\"\n        result = self._run(self.prelude)\n        if result[\"process_status\"] != \"completed\":\n            raise RuntimeError(f\"Local executor initialization failed: {result.get('stderr', '')[:500]}\")\n\n    def _read(self, deadline):\n        while b\"\\n\" not in self.buf:\n            wait = deadline - time.monotonic()\n            if wait <= 0:\n                return None\n            ready, _, _ = select.select([self.resp], [], [], wait)\n            if ready:\n                chunk = os.read(self.resp, 1 << 16)\n                if not chunk:\n                    return \"eof\"\n                self.buf += chunk\n        line, self.buf = self.buf.split(b\"\\n\", 1)\n        return json.loads(line)\n\n    def _kill(self):\n        if self.proc is not None:\n            try:\n                os.killpg(self.proc.pid, signal.SIGKILL)\n            except (ProcessLookupError, PermissionError):\n                pass\n            self.proc.wait()\n            try:\n                self.proc.stdin.close()\n            except Exception:\n                pass\n        if self.resp is not None:\n            os.close(self.resp)\n        self.proc = self.resp = None\n\n    def _run(self, code):\n        \"\"\"Run one cell; returns the result, or None if the process died or had to be killed.\"\"\"\n        self.last_failure = \"error\"\n        try:\n            self.proc.stdin.write(json.dumps({\"code\": code}) + \"\\n\")\n            self.proc.stdin.flush()\n        except (BrokenPipeError, OSError):\n            return None\n        result = self._read(time.monotonic() + self.timeout)\n        if result is None:  # timed out: interrupt, keep state if the cell stops\n            try:\n                self.proc.send_signal(signal.SIGINT)\n            except ProcessLookupError:\n                pass\n            self.last_failure = \"timeout\"\n            result = self._read(time.monotonic() + self.grace)\n            if isinstance(result, dict):\n                result[\"process_status\"] = \"timeout\"\n                return result\n            return None\n        return result if isinstance(result, dict) else None\n\n    def initialize(self):\n        self._spawn()\n\n    def execute(self, code):\n        if self.closed:\n            raise RuntimeError(\"Executor is closed\")\n        if self.proc is None:\n            self._spawn()\n        result = self._run(code)\n        if result is None:\n            # Died (memory limit, os._exit, segfault) or ignored SIGINT: restart with task data only.\n            self._kill()\n            self._spawn()\n            result = {\"process_status\": self.last_failure, \"stdout\": \"\", \"state_reset\": True,\n                      \"stderr\": (\"Cell did not stop after the timeout; process restarted.\" if self.last_failure == \"timeout\"\n                                 else \"Python process died (crash, exit or memory limit); process restarted.\")}\n        return result\n\n    def close(self):\n        if not self.closed:\n            self.closed = True\n            self._kill()\n\n    def __enter__(self):\n        try:\n            self.initialize()\n        except BaseException:\n            self.close()\n            raise\n        return self\n\n    def __exit__(self, *exc):\n        self.close()\n\n\ndef local_factory(timeout=30, max_output=16000, memory_mb=4096):\n    return lambda train, test_input: LocalExecutor(train, test_input, timeout, max_output, memory_mb)\n\n\ndef sandbox_factory(url, timeout=30, max_output=16000):\n    \"\"\"Docker sandbox sessions (evals/sandbox_client.py must be importable, i.e. evals/ on sys.path).\"\"\"\n    from sandbox_client import SandboxSession\n    return lambda train, test_input: SandboxSession(url, train, test_input, timeout, max_output)\n\n\n# ---------------------------------------------------------------- helpers shared with run_deepseek_tools.py\n\ndef tool_output(result, limit=16000):\n    text = result.get(\"stdout\", \"\")\n    if result.get(\"stderr\"):\n        text += \"\\n[stderr]\\n\" + result[\"stderr\"]\n    if result.get(\"state_reset\"):\n        text = \"[Python state lost. train/test_input/np restored; redefine other variables.]\\n\" + text\n    text = f\"[{result['process_status']}]\\n\" + (text or \"[no output; use print()]\")\n    return text if len(text) <= limit else text[:limit] + \"\\n[output truncated]\"\n\n\ndef submit_answer(session, code):\n    \"\"\"Evaluate the model's answer expression in its own session and return (feedback, grid|None).\n\n    `code` is an expression (e.g. solve(test_input)) or statements that set `answer`.\"\"\"\n    marker = f\"ARC_ANSWER_{uuid4().hex}:\"\n    cell = f'''\ndef _arc_answer():\n    import contextlib, io, json\n    _src = {code!r}\n    with contextlib.redirect_stdout(io.StringIO()):\n        try:\n            _expr = compile(_src, '<answer>', 'eval')\n        except SyntaxError:\n            _expr = None\n        if _expr is not None:\n            value = eval(_expr, globals())\n        else:\n            exec(compile(_src, '<answer>', 'exec'), globals())\n            if 'answer' not in globals():\n                raise NameError('code did not define answer')\n            value = globals()['answer']\n        if hasattr(value, 'tolist'):\n            value = value.tolist()\n        text = json.dumps(value, default=int)\n    print({marker!r} + text)\ntry:\n    _arc_answer()\nfinally:\n    del _arc_answer\n'''\n    result = session.execute(cell)\n    grid = None\n    if result[\"process_status\"] == \"completed\":\n        for line in reversed(result.get(\"stdout\", \"\").splitlines()):\n            if line.startswith(marker):\n                try:\n                    grid = json.loads(line[len(marker):])\n                except json.JSONDecodeError:\n                    pass\n                break\n        else:\n            return \"[answer not accepted] no value returned\", None\n    else:\n        return tool_output(result, session.max_output)[:2000] + \"\\n[answer not accepted]\", None\n    if not valid_grid(grid):\n        shape = (f\"{len(grid)} rows, row lengths {sorted({len(r) for r in grid if isinstance(r, list)})}\"\n                 if isinstance(grid, list) else type(grid).__name__)\n        return (\"[answer not accepted] must be a rectangular 1-30 by 1-30 grid of ints 0-9; got \" + shape), None\n    return f\"[answer accepted: {len(grid)}x{len(grid[0])} grid]\", grid\n\n\ndef check_answer_on_train(session, code):\n    \"\"\"Re-evaluate the submitted expression with `test_input` bound to each training input and compare with the\n    host's training outputs. None when there is nothing to check, else (failed_indices, feedback_text).\"\"\"\n    try:\n        expr = compile(code, \"<answer>\", \"eval\")\n    except SyntaxError:\n        return None\n    if \"test_input\" not in expr.co_names:\n        return None\n    marker = f\"ARC_CHECK_{uuid4().hex}:\"\n    inputs = [pair[\"input\"] for pair in session.train]\n    cell = f'''\ndef _arc_check():\n    import contextlib, io, json\n    _expr = compile({code!r}, '<answer>', 'eval')\n    outs = []\n    with contextlib.redirect_stdout(io.StringIO()):\n        for _inp in json.loads({json.dumps(inputs)!r}):\n            try:\n                v = eval(_expr, {{**globals(), 'test_input': _inp}})\n                if hasattr(v, 'tolist'):\n                    v = v.tolist()\n                outs.append(json.loads(json.dumps(v, default=int)))\n            except Exception as exc:\n                outs.append({{'error': type(exc).__name__ + ': ' + str(exc)[:200]}})\n    print({marker!r} + json.dumps(outs))\ntry:\n    _arc_check()\nfinally:\n    del _arc_check\n'''\n    result = session.execute(cell)\n    outs = None\n    if result[\"process_status\"] == \"completed\":\n        for line in reversed(result.get(\"stdout\", \"\").splitlines()):\n            if line.startswith(marker):\n                try:\n                    outs = json.loads(line[len(marker):])\n                except json.JSONDecodeError:\n                    pass\n                break\n    if not isinstance(outs, list) or len(outs) != len(inputs):\n        return None\n    failed, notes = [], []\n    for i, (o, pair) in enumerate(zip(outs, session.train)):\n        if o == pair[\"output\"]:\n            continue\n        failed.append(i)\n        if isinstance(o, dict) and \"error\" in o:\n            notes.append(f\"pair {i}: {o['error']}\")\n        elif valid_grid(o):\n            exp = pair[\"output\"]\n            if (len(o), len(o[0])) != (len(exp), len(exp[0])):\n                notes.append(f\"pair {i}: shape {len(o)}x{len(o[0])}, expected {len(exp)}x{len(exp[0])}\")\n            else:\n                notes.append(f\"pair {i}: {sum(a != b for ra, rb in zip(o, exp) for a, b in zip(ra, rb))} cells differ\")\n        else:\n            notes.append(f\"pair {i}: not a valid grid\")\n    return failed, \"; \".join(notes)\n", "evals/arc_harness/arcprize.py": "\"\"\"ARC Prize public-eval prompt and answer parser (github.com/arcprize/arc-agi-benchmarking), verbatim.\"\"\"\nimport json\nimport re\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[2]\nTASKS = ROOT / \"data/arc-agi-2-github/data/evaluation\"\nTEMPLATE = (ROOT / \"notes/prompts/arcprize_official_system_prompt.txt\").read_text()\n\n\ndef build_prompt(train, test_input):\n    ex = \"\"\n    for i, pair in enumerate(train):\n        ex += f\"--Example {i}-- \\n\\n INPUT: \\n\\n\" + json.dumps(pair[\"input\"]) + \"\\n\\n\"\n        ex += \"OUTPUT: \\n\\n\" + json.dumps(pair[\"output\"]) + \"\\n\\n\"\n    return TEMPLATE.format(training_examples=ex, test_input=json.dumps(test_input))\n\n\ndef is_grid(x):\n    return isinstance(x, list) and len(x) > 0 and all(isinstance(r, list) for r in x)\n\n\ndef extract_from_boxed(s):\n    m = re.search(r\"\\\\boxed\\{(.*?)\\}\", s, re.DOTALL)\n    if m:\n        try:\n            g = json.loads(m.group(1).strip())\n            return g if is_grid(g) else None\n        except json.JSONDecodeError:\n            return None\n    return None\n\n\ndef backscan_json_parser(s):\n    end = max(s.rfind(\"]\"), s.rfind(\"}\"))\n    if end < 0:\n        return None\n    close = s[end]\n    open_ = \"[\" if close == \"]\" else \"{\"\n    depth = 1\n    for i in range(end - 1, -1, -1):\n        depth += (s[i] == close) - (s[i] == open_)\n        if depth == 0:\n            try:\n                g = json.loads(s[i:end + 1])\n                return g if is_grid(g) else None\n            except json.JSONDecodeError:\n                return None\n    return None\n\n\ndef parse(s):\n    return extract_from_boxed(s) or backscan_json_parser(s or \"\")\n\n", "evals/arc_harness/grids.py": "\"\"\"ARC grid validity: the single definition used for answers, submissions and scoring.\"\"\"\n\n\ndef valid_grid(grid):\n    \"\"\"A rectangular list of 1-30 rows of 1-30 Python ints in 0-9.\"\"\"\n    return (isinstance(grid, list) and 1 <= len(grid) <= 30\n            and all(isinstance(row, list) and 1 <= len(row) <= 30 for row in grid)\n            and len({len(row) for row in grid}) == 1\n            and all(type(value) is int and 0 <= value <= 9 for row in grid for value in row))\n", "evals/arc_harness/prompts.py": "\"\"\"Everything the model reads: the opening message, tool descriptions and fixed instructions.\n\nKept byte-identical to harness persistent-v3 so results stay comparable across refactors.\n\"\"\"\nfrom .arcprize import build_prompt\n\n# Optional suffixes appended to the ARC Prize prompt (--prompt). \"plain\" keeps the original prompt.\nPROMPT_SUFFIXES = {\n    \"plain\": \"\",\n    \"lean\": (\"\\n\\nYou have a Python tool. Work efficiently: inspect the grids with code instead of reading them \"\n             \"by hand, form a hypothesis, and write code that computes the output. Never construct, copy or count \"\n             \"grids by hand. Check your rule on every training pair with code before answering, and if a pair \"\n             \"fails, fix the rule rather than explaining the mismatch away. Keep your reasoning brief.\"),\n    \"lean2\": (\"\\n\\nYou have a Python tool. The variables `train` (list of {'input', 'output'} grids) and \"\n              \"`test_input` are already loaded in it: use them directly and never retype or reassign them. \"\n              \"Work efficiently: inspect the grids with code instead of reading them by hand, form a hypothesis, \"\n              \"and write code that computes the output. Never construct, copy or count grids by hand. Check your \"\n              \"rule on every training pair with code before answering, and if a pair fails, fix the rule rather \"\n              \"than explaining the mismatch away. Before answering, check that the rule's assumptions also hold \"\n              \"on test_input. Keep your reasoning brief.\"),\n}\nCODE_MODE_SUFFIX = \"\\n\\nUse run_python to investigate; submit self-contained transform(grid) code with submit_solution.\"\nFINALIZE = \"Finalize now. Return only your best test output as a JSON grid.\"\nMODES = (\"grid\", \"code\")\n\n\ndef initial_messages(train, test_input, prompt, mode):\n    content = build_prompt(train, test_input) + PROMPT_SUFFIXES[prompt]\n    if mode == \"code\":\n        content += CODE_MODE_SUFFIX\n    return [{\"role\": \"user\", \"content\": content}]\n\n\ndef tool_specs(mode, cell_timeout):\n    descriptions = {\"run_python\": (\n        \"Execute a Python cell in your persistent IPython session. Variables/functions persist. \"\n        \"Preloaded: train (list of {'input': grid, 'output': grid}), test_input (grid), numpy as np. \"\n        f\"scipy is available. No internet or host files. Timeout {cell_timeout}s. \"\n        \"Use print() to inspect results. Check transformations on all training pairs.\"\n    )}\n    if mode == \"code\":\n        descriptions[\"submit_solution\"] = (\n            \"Submit self-contained Python code defining transform(grid), which returns a rectangular \"\n            \"grid of ints 0-9 (or numpy array). Include imports/helpers; execution uses a fresh namespace. \"\n            \"The harness checks every training example. Failures return feedback so you can revise. \"\n            \"If all pass, it applies transform to test_input and ends this attempt.\"\n        )\n    return [{\"type\": \"function\", \"function\": {\n        \"name\": name, \"description\": description,\n        \"parameters\": {\"type\": \"object\", \"properties\": {\"code\": {\"type\": \"string\"}},\n                       \"required\": [\"code\"], \"additionalProperties\": False},\n    }} for name, description in descriptions.items()]\n", "notes/prompts/arcprize_official_system_prompt.txt": "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\nBelow is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation in the training examples that maps the input to the output, then apply that pattern to the test input to give a final output.\n\nRespond in the format of the training output examples\n\n--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n--Test Input--\n{test_input}\n--End of Test Input--\n\nYour response:"}
for rel, text in BUNDLE.items():
    p = os.path.join('/kaggle/working/pkg', rel)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w').write(text)

In [ ]:

import sys, time, os, json, subprocess, glob
t=time.time()
wd = os.path.dirname(sorted(glob.glob('/kaggle/input/**/sglang-0.5.20-*.whl', recursive=True))[0])
r = subprocess.run([sys.executable,'-m','pip','install','--no-index','--find-links',wd,'-r',wd+'/requirements.lock'],capture_output=True,text=True)
print('offline install rc', r.returncode, 'in', round(time.time()-t), 's'); print(r.stdout[-600:], r.stderr[-1500:])
print(subprocess.run([sys.executable,'-c','import torch, sglang, numpy, scipy; print(torch.__version__, sglang.__version__, numpy.__version__, scipy.__version__)'],capture_output=True,text=True))
sys.path.insert(0, '/kaggle/working/pkg/evals')
from arc_harness.executor import LocalExecutor, submit_answer, tool_output
from arc_harness import agent_qwen
print('harness', agent_qwen.HARNESS_VERSION, 'cpus', os.cpu_count())
train=[{'input':[[1,2],[3,4]],'output':[[4,3],[2,1]]}]
with LocalExecutor(train, [[5,6],[7,8]], timeout=3) as ex:
    for code in ['import numpy as np, scipy.ndimage\nprint(np.rot90(np.array(train[0]["input"]),2).tolist())',
                 'x = np.zeros((3000,3000)); x.sum()', 'while True: pass', 'import os\nos._exit(3)',
                 'bytearray(6 * 2**30)', 'answer = np.rot90(np.array(test_input), 2)']:
        t=time.time(); res = ex.execute(code); print(round(time.time()-t,1),'s', json.dumps(res)[:300])
    print('submit:', submit_answer(ex, 'answer'))
# 16 concurrent executors (one per attempt)
from concurrent.futures import ThreadPoolExecutor
def one(i):
    with LocalExecutor(train, [[i%10]], timeout=10) as ex:
        return ex.execute('import numpy as np\nsum(range(10**6))')['process_status']
t=time.time()
with ThreadPoolExecutor(16) as pool: print(list(pool.map(one, range(16))), round(time.time()-t,1),'s for 16 parallel sessions')
print('loadavg', os.getloadavg())
